In [ ]:
import os
import csv
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings

warnings.filterwarnings('ignore')

from fontTools.ttLib import TTFont
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. КОНФИГУРАЦИЯ
# ============================================================

# Текущая директория (где находится скрипт)
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# Папка со шрифтами (относительно скрипта)
FONT_DIR = os.path.join(BASE_DIR, 'fonts')

# Файлы шрифтов (должны лежать в папке FONT_DIR)
FONT_FILES = [
    'Bukvarnaya.ttf', 'SchoolBook.ttf', 'Roboto.ttf', 'PT_Sans.ttf',
    'Oswald.ttf', 'Open_Sans.ttf', 'Lobster.ttf', 'Russo_One.ttf',
    'Kelly_Slab.ttf', 'Rubik_Mono_One.ttf', 'Pacifico.ttf', 'Fet.ttf',
    'Comic_Sans.ttf', 'Caveat.ttf', 'KS_Bistra.ttf', 'Great_Vibes.ttf',
    'WarmPixel.ttf', 'Gatchina.ttf'
]

# Целевая переменная (экспертные оценки)
TARGET = [1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]

# Анализируемые буквы (Unicode коды)
RUSSIAN_UNICODE = {
    'А': 0x0410, 'Б': 0x0411, 'В': 0x0412,
    'О': 0x041E, 'М': 0x041C, 'Р': 0x0420, 'У': 0x0423
}
LETTERS = list(RUSSIAN_UNICODE.keys())

# Пути для сохранения результатов (в текущей директории)
OUTPUT_CSV = os.path.join(BASE_DIR, 'features_multiglyph_final.csv')
MODEL_PATH = os.path.join(BASE_DIR, 'best_model.pkl')
SCALER_PATH = os.path.join(BASE_DIR, 'best_scaler.pkl')
FEATURES_PATH = os.path.join(BASE_DIR, 'best_features.txt')
PLOT_PATH = os.path.join(BASE_DIR, 'feature_importance.png')

# ============================================================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ ИЗВЛЕЧЕНИЯ ПРИЗНАКОВ
# ============================================================

def segment_length(p0, p1):
    """Евклидово расстояние между двумя точками (формула 2.1)"""
    return math.sqrt((p1[0] - p0[0])**2 + (p1[1] - p0[1])**2)


def bezier_quadratic(t, p0, p1, p2):
    """Квадратичная кривая Безье в точке t (формула 2.2)"""
    x = (1-t)**2 * p0[0] + 2*(1-t)*t * p1[0] + t**2 * p2[0]
    y = (1-t)**2 * p0[1] + 2*(1-t)*t * p1[1] + t**2 * p2[1]
    return (x, y)


def bezier_derivative(t, p0, p1, p2):
    """Первая производная квадратичной кривой Безье (формула 2.9)"""
    x = 2*(1-t)*(p1[0]-p0[0]) + 2*t*(p2[0]-p1[0])
    y = 2*(1-t)*(p1[1]-p0[1]) + 2*t*(p2[1]-p1[1])
    return (x, y)


def bezier_second_derivative(p0, p1, p2):
    """Вторая производная квадратичной кривой Безье (формула 2.10)"""
    x = 2*(p2[0] - 2*p1[0] + p0[0])
    y = 2*(p2[1] - 2*p1[1] + p0[1])
    return (x, y)


def curvature(t, p0, p1, p2):
    """Кривизна в точке t (формула 2.8)"""
    d1 = bezier_derivative(t, p0, p1, p2)
    d2 = bezier_second_derivative(p0, p1, p2)
    cross = abs(d1[0]*d2[1] - d1[1]*d2[0])
    norm = math.sqrt(d1[0]**2 + d1[1]**2)
    if norm < 1e-10:
        return 0.0
    return cross / (norm**3)


def arc_length_gauss_legendre(p0, p1, p2, n=8):
    """Дуговое расстояние методом Гаусса-Лежандра (формула 2.3)"""
    nodes = [-0.96028986, -0.79666648, -0.52553241, -0.18343464,
              0.18343464,  0.52553241,  0.79666648,  0.96028986]
    weights = [0.10122854, 0.22238103, 0.31370665, 0.36268378,
               0.36268378, 0.31370665, 0.22238103, 0.10122854]
    length = 0.0
    for i in range(n):
        t = 0.5 * nodes[i] + 0.5
        d = bezier_derivative(t, p0, p1, p2)
        speed = math.sqrt(d[0]**2 + d[1]**2)
        length += weights[i] * speed
    return 0.5 * length


def segment_area(p0, p1, p2):
    """Площадь сегмента через теорему Грина (формула 2.6)"""
    return (1/3) * abs((p1[0]-p0[0])*(p2[1]-p0[1]) - (p2[0]-p0[0])*(p1[1]-p0[1]))


def mean_curvature(p0, p1, p2, n=20):
    """Средняя кривизна по n точкам (формула 2.11)"""
    curvatures = []
    for i in range(n):
        t = i / (n - 1)
        curvatures.append(curvature(t, p0, p1, p2))
    return np.mean(curvatures)


def circle_deviation(p0, p2, p1, n=20):
    """Отклонение от эталонной окружности (формула 2.12)"""
    mx, my = (p0[0] + p2[0])/2, (p0[1] + p2[1])/2
    chord = segment_length(p0, p2)
    if chord < 1e-10:
        return 0.0
    dx, dy = p2[0] - p0[0], p2[1] - p0[1]
    h = abs(dx*(p1[1]-p0[1]) - dy*(p1[0]-p0[0])) / chord
    if h < 1e-10:
        return 0.0
    nx, ny = -dy/chord, dx/chord
    sign = 1 if (dx*(p1[1]-my) - dy*(p1[0]-mx)) > 0 else -1
    ideal_p1 = (mx + sign*h*nx, my + sign*h*ny)
    deviation = 0.0
    for i in range(n):
        t = i / (n - 1)
        real_pt = bezier_quadratic(t, p0, p1, p2)
        ideal_pt = bezier_quadratic(t, p0, ideal_p1, p2)
        deviation += (real_pt[0]-ideal_pt[0])**2 + (real_pt[1]-ideal_pt[1])**2
    return math.sqrt(deviation / n)


def count_inflection_points(segments):
    """Подсчёт точек перегиба (формула 2.13)"""
    curvatures = []
    for seg in segments:
        if seg['type'] == 'curve':
            curvatures.append(seg['mean_curvature'])
    count = 0
    for i in range(len(curvatures) - 1):
        if curvatures[i] * curvatures[i+1] < 0:
            count += 1
    return count


# ============================================================
# 3. PEN ДЛЯ СБОРА КОНТУРОВ ГЛИФА
# ============================================================

class GlyphPen:
    """Pen для сбора контуров глифа из TTF"""
    def __init__(self, glyph_set):
        self.glyph_set = glyph_set
        self.contours = []
        self._current_contour = []
        self._current_point = None

    def moveTo(self, pt):
        self._current_point = pt
        self._current_contour = []

    def lineTo(self, pt):
        if self._current_point is not None:
            self._current_contour.append(('line', [self._current_point, pt]))
        self._current_point = pt

    def curveTo(self, *points):
        pts = list(points)
        if self._current_point is not None:
            self._current_contour.append(('curve', [self._current_point, pts[0], pts[1]]))
        self._current_point = pts[-1]

    def qCurveTo(self, *points):
        pts = list(points)
        start = self._current_point
        i = 0
        while i < len(pts):
            if i + 1 < len(pts):
                cp = pts[i]
                end = pts[i+1]
                i += 2
            else:
                cp = pts[i]
                end = pts[i]
                i += 1
            self._current_contour.append(('curve', [(start[0], start[1]),
                                                      (cp[0], cp[1]),
                                                      (end[0], end[1])]))
            start = end
        self._current_point = pts[-1]

    def closePath(self):
        if self._current_contour:
            self.contours.append(self._current_contour)
        self._current_contour = []
        self._current_point = None

    def endPath(self):
        if self._current_contour:
            self.contours.append(self._current_contour)
        self._current_contour = []
        self._current_point = None

    def addComponent(self, glyphName, transformation):
        pass


# ============================================================
# 4. ФУНКЦИИ ИЗВЛЕЧЕНИЯ ПРИЗНАКОВ ИЗ ГЛИФА
# ============================================================

def find_glyph_name(font, unicode_code):
    """Находит имя глифа по Unicode-коду через cmap таблицу"""
    cmap = font.getBestCmap()
    return cmap.get(unicode_code)


def extract_glyph_features(font, glyph_name):
    """Извлечение геометрических признаков из одного глифа"""
    glyph_set = font.getGlyphSet()
    if glyph_name is None or glyph_name not in glyph_set:
        return None

    glyph = glyph_set[glyph_name]
    pen = GlyphPen(glyph_set)
    glyph.draw(pen)

    segments = []
    for contour in pen.contours:
        for item in contour:
            seg_type = item[0]
            points = item[1]
            if seg_type == 'line':
                p0, p1 = points[0], points[1]
                seg = {
                    'type': 'line',
                    'length': segment_length(p0, p1),
                    'p0': p0, 'p1': p1
                }
                segments.append(seg)
            elif seg_type == 'curve':
                p0, p1, p2 = points[0], points[1], points[2]
                s = arc_length_gauss_legendre(p0, p1, p2)
                l_chord = segment_length(p0, p2)
                a_segm = segment_area(p0, p1, p2)
                k_mean = mean_curvature(p0, p1, p2)
                delta_circ = circle_deviation(p0, p2, p1)
                seg = {
                    'type': 'curve',
                    'arc_length': s,
                    'chord_length': l_chord,
                    'r_curv': s / l_chord if l_chord > 0 else 1.0,
                    'a_norm': a_segm / (l_chord**2) if l_chord > 0 else 0.0,
                    'mean_curvature': k_mean,
                    'circle_deviation': delta_circ,
                    'p0': p0, 'p1': p1, 'p2': p2
                }
                segments.append(seg)

    line_segments = [s for s in segments if s['type'] == 'line']
    curve_segments = [s for s in segments if s['type'] == 'curve']

    features = {}
    features['total_line_length'] = sum(s['length'] for s in line_segments)
    features['total_arc_length'] = sum(s['arc_length'] for s in curve_segments)
    features['mean_r_curv'] = np.mean([s['r_curv'] for s in curve_segments]) if curve_segments else 0
    features['mean_a_norm'] = np.mean([s['a_norm'] for s in curve_segments]) if curve_segments else 0

    if curve_segments:
        total_arc = sum(s['arc_length'] for s in curve_segments)
        features['mean_curvature'] = sum(s['mean_curvature'] * s['arc_length'] for s in curve_segments) / total_arc if total_arc > 0 else 0
    else:
        features['mean_curvature'] = 0

    features['mean_circle_dev'] = np.mean([s['circle_deviation'] for s in curve_segments]) if curve_segments else 0
    features['inflection_points'] = count_inflection_points(segments)
    features['num_line_segments'] = len(line_segments)
    features['num_curve_segments'] = len(curve_segments)
    total_segments = len(segments)
    features['curve_ratio'] = len(curve_segments) / total_segments if total_segments > 0 else 0

    return features


# ============================================================
# 5. ОСНОВНОЙ ЦИКЛ ИЗВЛЕЧЕНИЯ ПРИЗНАКОВ
# ============================================================

def extract_all_features():
    """Извлечение признаков для всех шрифтов в папке"""
    print("ИЗВЛЕЧЕНИЕ ГЕОМЕТРИЧЕСКИХ ПРИЗНАКОВ")
    print(f"Папка со шрифтами: {FONT_DIR}")

    # Проверяем, существует ли папка
    if not os.path.exists(FONT_DIR):
        os.makedirs(FONT_DIR)
        print(f"Создана папка {FONT_DIR}")
        print("Пожалуйста, поместите файлы шрифтов в эту папку и запустите скрипт снова.")
        return None

    results = []

    for i, font_file in enumerate(FONT_FILES):
        font_path = os.path.join(FONT_DIR, font_file)

        if not os.path.exists(font_path):
            print(f"⊘ {font_file} --- ФАЙЛ НЕ НАЙДЕН")
            continue

        try:
            font = TTFont(font_path)
            all_glyph_features = {}
            processed_glyphs = 0
            glyph_map = {}

            for letter, code in RUSSIAN_UNICODE.items():
                gname = find_glyph_name(font, code)
                glyph_map[letter] = gname

            for letter, gname in glyph_map.items():
                if gname:
                    glyph_feat = extract_glyph_features(font, gname)
                    if glyph_feat is not None:
                        for key, value in glyph_feat.items():
                            all_glyph_features[f"{letter}_{key}"] = value
                        processed_glyphs += 1

            if processed_glyphs == 0:
                print(f"✗ {font_file} --- ни один глиф не обработан")
                continue

            all_glyph_features['font_name'] = font_file
            all_glyph_features['target'] = TARGET[i]
            results.append(all_glyph_features)

            print(f"✓ {font_file} --- обработано {processed_glyphs}/7 глифов")

        except Exception as e:
            print(f"✗ {font_file} --- ошибка: {e}")

    return results


def save_features_to_csv(results):
    """Сохранение признаков в CSV файл"""
    base_features = [
        'total_line_length', 'total_arc_length', 'mean_r_curv', 'mean_a_norm',
        'mean_curvature', 'mean_circle_dev', 'inflection_points',
        'num_line_segments', 'num_curve_segments', 'curve_ratio'
    ]

    all_fieldnames = ['font_name', 'target']
    for letter in LETTERS:
        for feat in base_features:
            all_fieldnames.append(f"{letter}_{feat}")

    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=all_fieldnames, extrasaction='ignore')
        writer.writeheader()
        for r in results:
            writer.writerow({k: r.get(k, 0.0) for k in all_fieldnames})

    print(f"ГОТОВО! Обработано шрифтов: {len(results)} из {len(FONT_FILES)}")
    print(f"Признаков на шрифт: {len(all_fieldnames) - 2}")
    print(f"Сохранено в: {OUTPUT_CSV}")

    return all_fieldnames


# ============================================================
# 6. ОБУЧЕНИЕ МОДЕЛИ
# ============================================================

def train_model():
    """Обучение модели случайного леса"""
    print("\nОБУЧЕНИЕ МОДЕЛИ СЛУЧАЙНОГО ЛЕСА")

    df = pd.read_csv(OUTPUT_CSV)
    print(f"Загружено шрифтов: {len(df)}")
    print(f"Всего столбцов: {len(df.columns)}")

    feature_cols = [c for c in df.columns if c not in ['font_name', 'target']]
    X = df[feature_cols].values
    y = df['target'].values
    font_names = df['font_name'].values

    print(f"Признаков: {len(feature_cols)}")
    print(f"Класс 1 (подходит): {sum(y)}")
    print(f"Класс 0 (не подходит): {len(y) - sum(y)}")

    # Стандартизация
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Кросс-валидация Leave-One-Out
    print("\nОптимальные гиперпараметры: n_estimators=50, max_depth=2, min_samples_leaf=3, class_weight='balanced'\n")

    loo = LeaveOneOut()
    rf = RandomForestClassifier(
        n_estimators=50,
        max_depth=2,
        min_samples_leaf=3,
        random_state=42,
        class_weight='balanced'
    )

    scores = cross_val_score(rf, X_scaled, y, cv=loo, scoring='accuracy')
    print(f"Точность кросс-валидации (Leave-One-Out): {np.mean(scores):.4f}")
    print(f"Стандартное отклонение: {np.std(scores):.4f}")

    # Обучение на полной выборке
    rf.fit(X_scaled, y)
    y_pred = rf.predict(X_scaled)
    print(f"\nAccuracy на обучающей выборке: {accuracy_score(y, y_pred):.4f}")

    print("\nМатрица ошибок:")
    cm = confusion_matrix(y, y_pred)
    print(cm)

    print("\nОтчёт о классификации:")
    print(classification_report(y, y_pred, target_names=['Не подходит', 'Подходит']))

    # Важность признаков
    importances = rf.feature_importances_
    indices = np.argsort(importances)[::-1]

    print("\nВАЖНОСТЬ ПРИЗНАКОВ (ТОП-15)")
    for rank, i in enumerate(indices[:15], 1):
        print(f"  {rank:2d}. {feature_cols[i]:45s}: {importances[i]:.4f}")

    # Визуализация
    plt.figure(figsize=(12, 8))
    top_n = 15
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, top_n))
    plt.barh(range(top_n), importances[indices[:top_n]], align='center', color=colors)
    plt.yticks(range(top_n), [feature_cols[i] for i in indices[:top_n]])
    plt.xlabel('Важность признака (Feature Importance)')
    plt.title('Топ-15 геометрических признаков по важности\n(модель Random Forest)', fontsize=14)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
    print(f"\nГрафик сохранён в: {PLOT_PATH}")

    # Сохранение модели
    joblib.dump(rf, MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)
    with open(FEATURES_PATH, 'w', encoding='utf-8') as f:
        for col in feature_cols:
            f.write(col + '\n')

    print(f"\nМОДЕЛЬ СОХРАНЕНА")
    print(f"Модель:          {MODEL_PATH}")
    print(f"Скейлер:         {SCALER_PATH}")
    print(f"Список признаков: {FEATURES_PATH}")
    print(f"Данные:          {OUTPUT_CSV}")
    print(f"График:          {PLOT_PATH}")

    return rf, scaler, feature_cols, df, y_pred


# ============================================================
# 7. ДЕМОНСТРАЦИОННАЯ ФУНКЦИЯ
# ============================================================

def demo_predict(font_path, rf, scaler, feature_cols):
    """
    Демонстрационная функция: загружает шрифт, извлекает признаки
    и выдаёт вердикт о пригодности для детских книг.
    """
    print(f"\nАНАЛИЗ ШРИФТА: {os.path.basename(font_path)}")

    try:
        font = TTFont(font_path)
        all_glyph_features = {}
        glyph_map = {}

        for letter, code in RUSSIAN_UNICODE.items():
            gname = find_glyph_name(font, code)
            glyph_map[letter] = gname

        for letter, gname in glyph_map.items():
            if gname:
                glyph_feat = extract_glyph_features(font, gname)
                if glyph_feat is not None:
                    for key, value in glyph_feat.items():
                        all_glyph_features[f"{letter}_{key}"] = value

        # Формируем вектор признаков
        X_new = np.array([[all_glyph_features.get(f, 0.0) for f in feature_cols]])
        X_new_scaled = scaler.transform(X_new)

        prediction = rf.predict(X_new_scaled)[0]
        proba = rf.predict_proba(X_new_scaled)[0]

        if prediction == 1:
            print(f"✓ РЕКОМЕНДУЕТСЯ для детских книг")
        else:
            print(f"✗ НЕ РЕКОМЕНДУЕТСЯ для детских книг")
        print(f"  Уверенность: P(не подходит)={proba[0]:.3f}, P(подходит)={proba[1]:.3f}")

        return prediction, proba

    except Exception as e:
        print(f"✗ Ошибка при анализе шрифта: {e}")
        return None, None


# ============================================================
# 8. ГЛАВНАЯ ФУНКЦИЯ
# ============================================================

def main():
    """Основная функция"""
    # Извлечение признаков
    results = extract_all_features()

    if results is None or len(results) == 0:
        print("Не удалось обработать ни одного шрифта.")
        return

    # Сохранение в CSV
    save_features_to_csv(results)

    # Обучение модели
    rf, scaler, feature_cols, df, y_pred = train_model()

    # Демонстрация работы на примерах
    print("\n" + "="*60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ МОДЕЛИ")
    print("="*60)

    # Проверяем на шрифтах из выборки
    for test_font in ['Roboto.ttf', 'Lobster.ttf', 'Bukvarnaya.ttf']:
        test_path = os.path.join(FONT_DIR, test_font)
        if os.path.exists(test_path):
            demo_predict(test_path, rf, scaler, feature_cols)


if __name__ == "__main__":
    main()